In [ ]:
"""
╔══════════════════════════════════════════════════════════════╗
║        GENERADOR DE DATOS SINTÉTICOS — DATENGEIST            ║
║        Heladería · México interior · 2 años de datos         ║
╚══════════════════════════════════════════════════════════════╝

Genera CSVs listos para cargar a PostgreSQL con tendencias reales:
  - Estacionalidad climática (México interior: lluvias jun-sep)
  - Correlación ventas ↔ temperatura ↔ lluvia
  - Festivos nacionales y locales con spike de ventas
  - Comportamiento de clientes por segmento
  - Detalle de ventas (~400k+ registros)
"""

import pandas as pd
import numpy as np
from faker import Faker
from tqdm import tqdm
import random
import math
import os
from datetime import date, timedelta, datetime

# ─────────────────────────────────────────────
#  CONFIG GENERAL
# ─────────────────────────────────────────────
SEED       = 42
FECHA_INI  = date(2023, 1, 1)
FECHA_FIN  = date(2024, 12, 31)
N_CLIENTES = 2_000
OUTPUT_DIR = "./csv_output"

random.seed(SEED)
np.random.seed(SEED)
fake = Faker("es_MX")
Faker.seed(SEED)

os.makedirs(OUTPUT_DIR, exist_ok=True)

fechas = pd.date_range(FECHA_INI, FECHA_FIN, freq="D")

# ─────────────────────────────────────────────
#  HELPERS DE ESTACIONALIDAD (México interior)
# ─────────────────────────────────────────────

def temperatura_diaria(fecha: date) -> float:
    """
    México interior (ej. CDMX / Querétaro / Aguascalientes):
    - Máximos de calor: mar-may (primavera seca)
    - Lluvias: jun-sep → bajan temperaturas
    - Mínimos: dic-ene
    Rango realista: 8°C – 32°C
    """
    dia_anio = fecha.timetuple().tm_yday
    # Sinusoide base: máximo en mayo (~día 135), mínimo en enero
    base = 20 + 7 * math.sin(2 * math.pi * (dia_anio - 30) / 365)
    # Efecto lluvia: baja ~4°C en temporada de lluvia
    lluvia_factor = -4 * max(0, math.sin(2 * math.pi * (dia_anio - 150) / 150))
    ruido = np.random.normal(0, 1.2)
    return round(max(8.0, min(35.0, base + lluvia_factor + ruido)), 2)


def precipitacion_diaria(fecha: date) -> float:
    """
    Temporada de lluvias jun-sep, casi cero el resto del año.
    """
    mes = fecha.month
    if mes in (6, 7, 8, 9):
        # 60% días con lluvia en temporada
        if random.random() < 0.60:
            return round(np.random.exponential(scale=12), 2)
    elif mes in (5, 10):
        if random.random() < 0.15:
            return round(np.random.exponential(scale=4), 2)
    return 0.0


FESTIVOS_MEXICO = {
    # (mes, dia): descripción
    (1, 1):  "Año Nuevo",
    (2, 5):  "Día de la Constitución",
    (3, 21): "Natalicio de Juárez",
    (4, 30): "Día del Niño",
    (5, 1):  "Día del Trabajo",
    (5, 10): "Día de las Madres",
    (5, 15): "Día del Maestro",
    (9, 15): "Víspera de Independencia",
    (9, 16): "Día de Independencia",
    (10, 31):"Halloween",
    (11, 1): "Día de Muertos",
    (11, 2): "Día de Muertos",
    (11, 20):"Revolución Mexicana",
    (12, 12):"Virgen de Guadalupe",
    (12, 24):"Nochebuena",
    (12, 25):"Navidad",
    (12, 31):"Fin de Año",
}

# Semana Santa (varía por año)
SEMANA_SANTA = {
    2023: [date(2023, 4, 3), date(2023, 4, 4), date(2023, 4, 5),
           date(2023, 4, 6), date(2023, 4, 7)],
    2024: [date(2024, 3, 25), date(2024, 3, 26), date(2024, 3, 27),
           date(2024, 3, 28), date(2024, 3, 29)],
}
SEMANA_SANTA_SET = set(d for lst in SEMANA_SANTA.values() for d in lst)


def evento_festivo(fecha: date) -> str:
    if fecha in SEMANA_SANTA_SET:
        return "Semana Santa"
    clave = (fecha.month, fecha.day)
    return FESTIVOS_MEXICO.get(clave, "")


def multiplicador_ventas(temp: float, precip: float, festivo: str, dia_semana: int) -> float:
    """
    Factores que afectan el volumen de ventas:
    - Temperatura alta → más ventas
    - Lluvia fuerte → menos ventas (gente no sale)
    - Festivo → spike importante
    - Fin de semana → más ventas
    """
    factor = 1.0
    # Temperatura: por cada grado sobre 20°C sube 3%
    factor *= 1 + max(0, (temp - 20) * 0.03)
    # Lluvia: >10mm baja 20%, >25mm baja 40%
    if precip > 25:
        factor *= 0.60
    elif precip > 10:
        factor *= 0.80
    # Festivo: +50% a +120%
    if festivo:
        factor *= random.uniform(1.5, 2.2)
    # Fin de semana (sáb=5, dom=6)
    if dia_semana in (5, 6):
        factor *= 1.30
    elif dia_semana == 4:  # viernes
        factor *= 1.15
    return factor


# ─────────────────────────────────────────────
#  1. VARIABLES EXTERNAS
# ─────────────────────────────────────────────
print("⏳ Generando Variables_Externas...")

vars_externas = []
for f in tqdm(fechas):
    temp   = temperatura_diaria(f.date())
    precip = precipitacion_diaria(f.date())
    festivo = evento_festivo(f.date())
    vars_externas.append({
        "Fecha":                   f.date(),
        "Temperatura":             temp,
        "Precipitacion":           precip,
        "Eventos_Festivos_Locales": festivo,
    })

df_vars = pd.DataFrame(vars_externas)
df_vars.to_csv(f"{OUTPUT_DIR}/Variables_Externas.csv", index=False)
print(f"   ✅ {len(df_vars)} filas → Variables_Externas.csv")


# ─────────────────────────────────────────────
#  2. DIM_PRODUCTOS_Y_SABORES
# ─────────────────────────────────────────────
print("⏳ Generando Dim_Productos_y_Sabores...")

SABORES = [
    "Vainilla", "Chocolate", "Fresa", "Mango", "Limón",
    "Pistache", "Nuez", "Cajeta", "Guanábana", "Mamey",
    "Tamarindo", "Coco", "Frambuesa", "Elote", "Tequila",
]

CATEGORIAS_HELADO = [
    ("Artesanal",       18.0, 45.0),   # (categoria, costo_lt, precio_sug)
    ("Premium",         25.0, 65.0),
    ("Bajo en azúcar",  22.0, 55.0),
    ("Niños",           12.0, 30.0),
    ("Temporada",       20.0, 50.0),
]

INGREDIENTES = {
    "Vainilla":    "leche, crema, azúcar, vainilla natural, yemas de huevo",
    "Chocolate":   "leche, crema, cacao 70%, azúcar, lecitina de soya",
    "Fresa":       "leche, crema, fresa fresca, azúcar, jugo de limón",
    "Mango":       "leche, crema, pulpa de mango ataulfo, azúcar",
    "Limón":       "leche, crema, jugo de limón, ralladura, azúcar",
    "Pistache":    "leche, crema, pasta de pistache, azúcar, sal de mar",
    "Nuez":        "leche, crema, nuez pecana tostada, caramelo, azúcar",
    "Cajeta":      "leche de cabra, azúcar, canela, vainilla",
    "Guanábana":   "leche, crema, pulpa de guanábana, azúcar, limón",
    "Mamey":       "leche, crema, pulpa de mamey, azúcar, canela",
    "Tamarindo":   "leche, crema, concentrado de tamarindo, chile piquín, azúcar",
    "Coco":        "leche de coco, crema, coco rallado, azúcar, vainilla",
    "Frambuesa":   "leche, crema, frambuesa, azúcar, pectina natural",
    "Elote":       "leche, crema, elote dulce, azúcar, mantequilla",
    "Tequila":     "leche, crema, tequila reposado, limón, sal de gusano, azúcar",
}

productos = []
pid = 1
for sabor in SABORES:
    for cat, costo, precio in CATEGORIAS_HELADO:
        ruido_c = round(random.uniform(-2, 2), 2)
        ruido_p = round(random.uniform(-3, 3), 2)
        productos.append({
            "ID_Producto":    pid,
            "Categoria":      cat,
            "costo_prod_lt":  round(costo + ruido_c, 2),
            "precio_sug":     round(precio + ruido_p, 2),
            "ingredientes":   INGREDIENTES.get(sabor, "leche, crema, azúcar"),
        })
        pid += 1

df_prod = pd.DataFrame(productos)
df_prod.to_csv(f"{OUTPUT_DIR}/Dim_Productos_y_Sabores.csv", index=False)
print(f"   ✅ {len(df_prod)} filas → Dim_Productos_y_Sabores.csv")


# ─────────────────────────────────────────────
#  3. DIM_CLIENTES_Y_SEGMENTOS
# ─────────────────────────────────────────────
print("⏳ Generando Dim_Clientes_y_Segmentos...")

GIROS = ["Familiar", "Corporativo", "Revendedor", "Cafetería", "Restaurante", "Particular"]
SEGMENTOS = ["VIP", "Frecuente", "Ocasional", "Nuevo"]
FRECUENCIAS = ["Semanal", "Quincenal", "Mensual", "Esporádico"]
UBICACIONES_MX = [
    "Colonia Centro", "Colonia Roma", "Colonia Condesa", "Polanco",
    "Coyoacán", "Xochimilco", "Tlalpan", "Iztapalapa",
    "Naucalpan", "Ecatepec", "Tlalnepantla", "Toluca",
    "Querétaro Centro", "Aguascalientes Norte", "León Guanajuato",
]

SEG_TICKET = {
    "VIP":       (300, 800),
    "Frecuente": (150, 400),
    "Ocasional": ( 80, 250),
    "Nuevo":     ( 50, 150),
}

clientes = []
for cid in range(1, N_CLIENTES + 1):
    seg  = random.choices(SEGMENTOS, weights=[0.10, 0.30, 0.40, 0.20])[0]
    giro = random.choices(GIROS, weights=[0.35, 0.15, 0.10, 0.15, 0.10, 0.15])[0]
    t_min, t_max = SEG_TICKET[seg]
    clientes.append({
        "ID_Cliente":        cid,
        "Nombre":            fake.company() if giro in ("Corporativo","Cafetería","Restaurante","Revendedor") else fake.name(),
        "Giro":              giro,
        "Segmento":          seg,
        "frecuencia_compra": random.choices(FRECUENCIAS, weights=[0.20, 0.25, 0.35, 0.20])[0],
        "ticket_prom":       round(random.uniform(t_min, t_max), 2),
        "sabor_preferido":   random.choice(SABORES),
        "ubicacion":         random.choice(UBICACIONES_MX),
    })

df_cli = pd.DataFrame(clientes)
df_cli.to_csv(f"{OUTPUT_DIR}/Dim_Clientes_y_Segmentos.csv", index=False)
print(f"   ✅ {len(df_cli)} filas → Dim_Clientes_y_Segmentos.csv")


# ─────────────────────────────────────────────
#  4 & 5. VENTAS + DETALLE_VENTAS  (el grueso)
# ─────────────────────────────────────────────
print("⏳ Generando Ventas y Detalle_ventas (esto tarda ~1-3 min)...")

METODOS_PAGO    = ["Efectivo", "Tarjeta débito", "Tarjeta crédito", "Transferencia", "QR"]
TIPOS_VENTA     = ["Mostrador", "Para llevar", "Pedido online", "Mayoreo"]
SABOR_IDS       = list(range(1, len(SABORES) + 1))
PRODUCTO_IDS    = df_prod["ID_Producto"].tolist()
CLIENTE_IDS     = df_cli["ID_Cliente"].tolist()

# Construir lookup de temperatura/precipitacion/festivo por fecha
vars_lookup = df_vars.set_index("Fecha")

ventas_rows   = []
detalle_rows  = []
ticket_id     = 1
VENTAS_BASE   = 550    # tickets promedio por día en temporada normal

for f in tqdm(fechas):
    fdate      = f.date()
    row_ext    = vars_lookup.loc[fdate]
    temp       = float(row_ext["Temperatura"])
    precip     = float(row_ext["Precipitacion"])
    festivo    = str(row_ext["Eventos_Festivos_Locales"])
    dia_semana = f.dayofweek

    mult       = multiplicador_ventas(temp, precip, festivo, dia_semana)
    n_tickets  = max(50, int(np.random.poisson(VENTAS_BASE * mult)))

    # Distribución de horarios: picos 12-15h y 17-20h
    horas_prob = [0]*8 + [0.01]*2 + [0.04, 0.07, 0.10, 0.12, 0.10,
                   0.09, 0.08, 0.10, 0.10, 0.09, 0.06, 0.03, 0.01] + [0]*1
    # normalizar
    suma_h = sum(horas_prob)
    horas_prob = [p/suma_h for p in horas_prob]

    for _ in range(n_tickets):
        hora  = np.random.choice(range(24), p=horas_prob)
        mins  = random.randint(0, 59)
        segs  = random.randint(0, 59)
        ts    = datetime(fdate.year, fdate.month, fdate.day, hora, mins, segs)

        cid   = random.choice(CLIENTE_IDS)
        pago  = random.choices(METODOS_PAGO, weights=[0.35, 0.25, 0.20, 0.10, 0.10])[0]

        # Número de productos por ticket (1-6, mayoría 1-2)
        n_prods = random.choices([1, 2, 3, 4, 5, 6], weights=[0.40, 0.30, 0.15, 0.08, 0.04, 0.03])[0]
        prods_ticket = random.sample(PRODUCTO_IDS, min(n_prods, len(PRODUCTO_IDS)))

        total = 0.0
        for pid_t in prods_ticket:
            prod_row = df_prod[df_prod["ID_Producto"] == pid_t].iloc[0]
            sabor_id = random.choice(SABOR_IDS)
            cantidad = random.choices([1, 2, 3, 4], weights=[0.55, 0.28, 0.12, 0.05])[0]
            # Precio con pequeña variación respecto al sugerido
            precio_u = round(prod_row["precio_sug"] * random.uniform(0.95, 1.05), 2)
            tipo_v   = random.choices(TIPOS_VENTA, weights=[0.55, 0.25, 0.12, 0.08])[0]
            subtotal = round(precio_u * cantidad, 2)
            total   += subtotal

            detalle_rows.append({
                "ID_Ticket":      ticket_id,
                "ID_Producto":    pid_t,
                "ID_Sabor":       sabor_id,
                "Cantidad":       cantidad,
                "Precio_Unitario":precio_u,
                "Tipo_Venta":     tipo_v,
            })

        ventas_rows.append({
            "ID_Ticket":      ticket_id,
            "Timestamp":      ts,
            "ID_Cliente":     cid,
            "payment_method": pago,
            "total":          round(total, 2),
        })
        ticket_id += 1

    # Volcado parcial cada 90 días para no saturar RAM
    if len(ventas_rows) >= 90_000:
        pd.DataFrame(ventas_rows).to_csv(
            f"{OUTPUT_DIR}/Ventas.csv",
            mode="a", index=False,
            header=not os.path.exists(f"{OUTPUT_DIR}/Ventas.csv") or os.path.getsize(f"{OUTPUT_DIR}/Ventas.csv") == 0
        )
        ventas_rows = []

    if len(detalle_rows) >= 250_000:
        pd.DataFrame(detalle_rows).to_csv(
            f"{OUTPUT_DIR}/Detalle_ventas.csv",
            mode="a", index=False,
            header=not os.path.exists(f"{OUTPUT_DIR}/Detalle_ventas.csv") or os.path.getsize(f"{OUTPUT_DIR}/Detalle_ventas.csv") == 0
        )
        detalle_rows = []

# Volcar lo que quedó en buffer
if ventas_rows:
    v_header = not os.path.exists(f"{OUTPUT_DIR}/Ventas.csv") or os.path.getsize(f"{OUTPUT_DIR}/Ventas.csv") == 0
    pd.DataFrame(ventas_rows).to_csv(f"{OUTPUT_DIR}/Ventas.csv", mode="a", index=False, header=v_header)
if detalle_rows:
    d_header = not os.path.exists(f"{OUTPUT_DIR}/Detalle_ventas.csv") or os.path.getsize(f"{OUTPUT_DIR}/Detalle_ventas.csv") == 0
    pd.DataFrame(detalle_rows).to_csv(f"{OUTPUT_DIR}/Detalle_ventas.csv", mode="a", index=False, header=d_header)

n_ventas  = ticket_id - 1
print(f"   ✅ {n_ventas:,} tickets → Ventas.csv")
print(f"   ✅ Detalle_ventas.csv generado")


# ─────────────────────────────────────────────
#  6. OPERACIONES_Y_PERSONAL
# ─────────────────────────────────────────────
print("⏳ Generando Operaciones_y_Personal...")

TURNOS = ["Matutino", "Vespertino", "Nocturno"]
TURNO_HORAS = {"Matutino": 6, "Vespertino": 8, "Nocturno": 4}  # horas del turno

ops_rows = []
for f in tqdm(fechas):
    fdate   = f.date()
    row_ext = vars_lookup.loc[fdate]
    temp    = float(row_ext["Temperatura"])
    festivo = str(row_ext["Eventos_Festivos_Locales"])
    dia_sem = f.dayofweek
    es_finde = dia_sem in (5, 6)

    for turno in TURNOS:
        # Más empleados en temporada alta y fines de semana
        base_emp = 4 if turno == "Vespertino" else 3
        if es_finde or festivo:
            base_emp += random.randint(1, 3)
        if temp > 28:
            base_emp += 1
        n_emp = max(2, base_emp + random.randint(-1, 1))

        costo_h = round(random.uniform(42.0, 65.0), 2)  # MXN/hora, rango real 2023-24

        pago = random.choices(METODOS_PAGO, weights=[0.35, 0.25, 0.20, 0.10, 0.10])[0]

        ops_rows.append({
            "Fecha":          fdate,
            "Turno":          turno,
            "Num_empleados":  n_emp,
            "payment_method": pago,
            "costo_hora":     costo_h,
        })

df_ops = pd.DataFrame(ops_rows)
df_ops.to_csv(f"{OUTPUT_DIR}/Operaciones_y_Personal.csv", index=False)
print(f"   ✅ {len(df_ops)} filas → Operaciones_y_Personal.csv")


# ─────────────────────────────────────────────
#  RESUMEN FINAL
# ─────────────────────────────────────────────
print("\n" + "═"*55)
print("  RESUMEN DE ARCHIVOS GENERADOS")
print("═"*55)
archivos = [
    "Variables_Externas.csv",
    "Dim_Productos_y_Sabores.csv",
    "Dim_Clientes_y_Segmentos.csv",
    "Ventas.csv",
    "Detalle_ventas.csv",
    "Operaciones_y_Personal.csv",
]
total_mb = 0
for arch in archivos:
    ruta = f"{OUTPUT_DIR}/{arch}"
    if os.path.exists(ruta):
        mb = os.path.getsize(ruta) / 1_048_576
        total_mb += mb
        filas = sum(1 for _ in open(ruta)) - 1
        print(f"  📄 {arch:<35} {filas:>10,} filas   {mb:>6.1f} MB")
    else:
        print(f"  ⚠️  {arch} — no encontrado")
print("─"*55)
print(f"  {'TOTAL':>46}  {total_mb:>6.1f} MB")
print("═"*55)
print("\n  CSVs listos en ./csv_output/")

⏳ Generando Variables_Externas...


100%|██████████| 731/731 [00:00<00:00, 61130.00it/s]


   ✅ 731 filas → Variables_Externas.csv
⏳ Generando Dim_Productos_y_Sabores...
   ✅ 75 filas → Dim_Productos_y_Sabores.csv
⏳ Generando Dim_Clientes_y_Segmentos...
   ✅ 2000 filas → Dim_Clientes_y_Segmentos.csv
⏳ Generando Ventas y Detalle_ventas (esto tarda ~1-3 min)...


100%|██████████| 731/731 [08:47<00:00,  1.39it/s]


   ✅ 479,986 tickets → Ventas.csv
   ✅ Detalle_ventas.csv generado
⏳ Generando Operaciones_y_Personal...


100%|██████████| 731/731 [00:00<00:00, 6440.54it/s]

   ✅ 2193 filas → Operaciones_y_Personal.csv

═══════════════════════════════════════════════════════
  RESUMEN DE ARCHIVOS GENERADOS
═══════════════════════════════════════════════════════
  📄 Variables_Externas.csv                     731 filas      0.0 MB
  📄 Dim_Productos_y_Sabores.csv                 75 filas      0.0 MB
  📄 Dim_Clientes_y_Segmentos.csv             2,000 filas      0.2 MB


  📄 Ventas.csv                             479,986 filas     23.3 MB
  📄 Detalle_ventas.csv                   1,030,726 filas     31.2 MB
  📄 Operaciones_y_Personal.csv               2,193 filas      0.1 MB
───────────────────────────────────────────────────────
                                           TOTAL    54.9 MB
═══════════════════════════════════════════════════════

  CSVs listos en ./csv_output/
  Siguiente paso: python scripts/csv_to_postgres.py



In [5]:
import pandas as pd
from sqlalchemy import create_engine

PG_USER     = "admin"
PG_PASSWORD = "cuyos123"
PG_HOST     = "localhost"
PG_PORT     = "5432"
PG_DB       = "datengeist"

engine = create_engine(
    f"postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}"
)

# Orden respetando foreign keys
TABLAS = [
    ("./csv_output/Variables_Externas.csv",       "variables_externas"),
    ("./csv_output/Dim_Productos_y_Sabores.csv",  "dim_productos_y_sabores"),
    ("./csv_output/Dim_Clientes_y_Segmentos.csv", "dim_clientes_y_segmentos"),
    ("./csv_output/Ventas.csv",                   "ventas"),
    ("./csv_output/Detalle_ventas.csv",            "detalle_ventas"),
    ("./csv_output/Operaciones_y_Personal.csv",   "operaciones_y_personal"),
]

for csv_path, table_name in TABLAS:
    print(f"⏳ Cargando {table_name}...")
    # chunksize para los archivos grandes (Ventas, Detalle)
    for chunk in pd.read_csv(csv_path, chunksize=50_000):
        chunk.columns = chunk.columns.str.lower()
        chunk.to_sql(table_name, engine, if_exists="append", index=False)
    print(f"   ✅ {table_name} lista")

print("\n🎉 Todas las tablas cargadas en PostgreSQL")

⏳ Cargando variables_externas...
   ✅ variables_externas lista
⏳ Cargando dim_productos_y_sabores...
   ✅ dim_productos_y_sabores lista
⏳ Cargando dim_clientes_y_segmentos...
   ✅ dim_clientes_y_segmentos lista
⏳ Cargando ventas...
   ✅ ventas lista
⏳ Cargando detalle_ventas...
   ✅ detalle_ventas lista
⏳ Cargando operaciones_y_personal...
   ✅ operaciones_y_personal lista

🎉 Todas las tablas cargadas en PostgreSQL
